<a href="https://colab.research.google.com/github/felipeafigueiredo/advanced_rag/blob/main/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# [Célula 1] Ambiente: instala dependências, sobe o Ollama na GPU do Colab e baixa os modelos
!apt-get update -qq && apt-get install -y -qq zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
!pip -q install ollama rank_bm25 pandas numpy

import subprocess, time
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)

!ollama pull nomic-embed-text
!ollama pull qwen2.5:3b
print("Ambiente pronto!")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


Ambiente pronto!


In [ ]:
# [Célula 2] Clonar repositório e carregar corpus (.md) e questions.json
import os, re, glob, json, time, shutil, unicodedata
import numpy as np
import pandas as pd
import ollama
from rank_bm25 import BM25Okapi

REPO_URL = "https://github.com/felipeafigueiredo/advanced_rag.git"
REPO_DIR = "advanced_rag"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

CORPUS_DIR = f"{REPO_DIR}/corpus"
QUESTIONS_PATH = f"{REPO_DIR}/questions.json"
RESULTS_DIR = f"{REPO_DIR}/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

SOURCE_ALIASES = {
    "regulamento_graduacao": "regulamento dos cursos",
    "res_013_2022": "reocupa",
    "in_05_2022": "grau individual",
    "in_08_2023": "deposito do trabalho",
    "in_10_2025": "diplomas digitais",
    "in_11_2026": "revalidacao",
}

def norm(s):
    return unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode().lower()

def get_doc_id(path):
    name = norm(os.path.basename(path))
    for doc_id, marker in SOURCE_ALIASES.items():
        if marker in name:
            return doc_id
    return os.path.basename(path)

def clean_md(text):
    text = re.sub(r"<!--.*?-->", "", text, flags=re.S)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

docs = {}
for path in sorted(glob.glob(f"{CORPUS_DIR}/*.md")):
    doc_id = get_doc_id(path)
    with open(path, encoding="utf-8") as f:
        docs[doc_id] = {"path": path, "text": clean_md(f.read())}

questions = json.load(open(QUESTIONS_PATH, encoding="utf-8"))
print(f"Documentos carregados ({len(docs)}):", list(docs.keys()))
print(f"Perguntas carregadas: {len(questions)}")

Documentos carregados (6): ['in_10_2025', 'in_11_2026', 'in_05_2022', 'in_08_2023', 'res_013_2022', 'regulamento_graduacao']
Perguntas carregadas: 20


In [ ]:
# ==================== CÉLULA 3 (código) ====================
# [Célula 3] Funções de Embedding, Geração (RAG) com Warm-up e Avaliação Automática
GEN_MODEL = "qwen2.5:3b"
EMB_MODEL = "nomic-embed-text"
TOP_K = 6

# Prompt v2: sem abertura fixa ("Não, pois..."), texto corrido, sem listas e sem inventar artigos.
SYSTEM_PROMPT = (
    "Você é um assistente especialista nas normas acadêmicas da UFRN. "
    "Responda à pergunta usando EXCLUSIVAMENTE as informações presentes nos trechos de contexto fornecidos. "
    "Reúna todos os detalhes, prazos, percentuais, pesos e exceções dos trechos que sejam pertinentes à pergunta. "
    "Comece diretamente pela resposta; use 'Sim' ou 'Não' no início apenas se a pergunta for de sim ou não. "
    "Escreva em texto corrido, em um ou dois parágrafos, sem listas, tópicos, marcadores, travessões iniciais ou numeração. "
    "Cite o número de um artigo somente quando ele aparecer literalmente no trecho utilizado; nunca invente artigos. "
    "Somente se nenhum trecho tiver relação com a pergunta, responda apenas: INFORMAÇÃO AUSENTE."
)

def embed_texts(texts):
    res = ollama.embed(model=EMB_MODEL, input=texts)
    arr = np.array(res["embeddings"], dtype="float32")
    norms = np.linalg.norm(arr, axis=1, keepdims=True)
    return arr / np.clip(norms, 1e-12, None)

# Cacheia o embedding das 20 perguntas e aquece o LLM na GPU
QUESTION_VECS = {q["id"]: embed_texts([q["pergunta"]])[0] for q in questions}
_ = ollama.generate(model=GEN_MODEL, prompt="Olá", options={"num_predict": 2})
print("Warm-up da GPU e embeddings das perguntas concluídos!")

def clean_answer(text):
    """Garante resposta em linha única (CSV limpo): remove marcadores de lista/markdown e quebras de linha."""
    lines = []
    for ln in text.splitlines():
        ln = re.sub(r"^\s*(?:[-–—•*]|#{1,6}|\d+[.)])\s+", "", ln).replace("**", "").replace("`", "").strip()
        if ln:
            lines.append(ln)
    if not lines:
        return ""
    out = lines[0]
    for ln in lines[1:]:
        out += (" " if out[-1] in ".;:!?" else "; ") + ln
    return re.sub(r"\s+", " ", out).strip()

def generate_answer(question, hits, seed=42):
    ctx = "\n\n".join(f"[{i+1}] Fonte: {h['doc_id']}\n{h['text']}" for i, h in enumerate(hits))
    prompt = (f"Contexto:\n{ctx}\n\nPergunta: {question}\n\n"
              "Resposta objetiva e completa, em texto corrido (cite artigos e valores presentes no contexto):")
    t0 = time.perf_counter()
    res = ollama.generate(
        model=GEN_MODEL, system=SYSTEM_PROMPT, prompt=prompt,
        options={"temperature": 0.2, "seed": seed, "num_predict": 350}
    )
    return clean_answer(res["response"]), time.perf_counter() - t0

def grade_answer(question, gold, pred):
    """Retorna 0 imediatamente em caso de recusa e avalia os dados essenciais contra o gabarito."""
    pred_n = norm(pred)
    if not pred_n or "informacao ausente" in pred_n or "nao encontr" in pred_n or "nao ha informa" in pred_n:
        return 0
    prompt = (
        f"Pergunta: {question}\n"
        f"Gabarito Oficial: {gold}\n"
        f"Resposta Avaliada: {pred}\n\n"
        "Critério: A Resposta Avaliada acerta os números, prazos, percentuais ou regras centrais do Gabarito Oficial sem inventar dados contraditórios?\n"
        "Responda APENAS com uma palavra: 'sim' ou 'nao'."
    )
    res = ollama.generate(model=GEN_MODEL, prompt=prompt, options={"temperature": 0.0, "num_predict": 5})
    ans = norm(res["response"])
    return 1 if "sim" in ans else 0

Warm-up da GPU e embeddings das perguntas concluídos!


In [ ]:
# [Célula 4] Construção dos Índices: Baseline e Família F2
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

PT_STOPWORDS = {
    "a", "o", "as", "os", "um", "uma", "de", "do", "da", "dos", "das", "em", "no", "na", "nos", "nas",
    "por", "para", "com", "sem", "sob", "sobre", "e", "ou", "que", "se", "ao", "aos", "pelo", "pela",
    "como", "qual", "quais", "sao", "foi", "ser", "ter", "seu", "sua", "seus", "suas", "esse", "essa"
}

def tokenize_pt(text):
    return [w for w in re.findall(r"\w+", norm(text)) if w not in PT_STOPWORDS and len(w) > 1]

# 1. BASELINE: Chunking de tamanho fixo (sofre com 'Arbitrary cuts')
def build_fixed_chunks(docs, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    out = []
    for doc_id, d in docs.items():
        text = re.sub(r"\s+", " ", d["text"]).strip()
        start, idx, n = 0, 0, len(text)
        while start < n:
            end = min(start + size, n)
            if end < n:
                cut = text.rfind(" ", start + int(size * 0.8), end)
                if cut != -1: end = cut
            out.append({"chunk_id": f"{doc_id}#{idx}", "doc_id": doc_id, "text": text[start:end].strip()})
            idx += 1
            if end >= n: break
            start = max(end - overlap, start + 1)
    return out

# 2. F2 TÉCNICA A - SEMANTIC CHUNKING: Agrupa unidades normativas e corta na queda de similaridade (Slide 11)
def split_into_units(text, unit_size=280):
    raw_parts = [p.strip() for p in re.split(r"(?=\bArt\.\s*\d+|\bCAPÍTULO\b|\bSeção\b|\bANEXO\b)|\n+", text) if p.strip()]
    units, buf = [], ""
    for p in raw_parts:
        if len(buf) + len(p) + 1 <= unit_size:
            buf = f"{buf} {p}".strip()
        else:
            if buf: units.append(buf)
            while len(p) > unit_size * 2:
                cut = p.rfind(" ", unit_size, unit_size * 2)
                if cut == -1: cut = unit_size * 2
                units.append(p[:cut].strip())
                p = p[cut:].strip()
            buf = p
    if buf: units.append(buf)
    return units

def build_semantic_chunks(docs, sim_threshold=0.68, min_chars=650, max_chars=1350):
    out = []
    for doc_id, d in docs.items():
        units = split_into_units(d["text"])
        if not units: continue
        vecs = embed_texts(units)
        curr, idx = units[0], 0
        for i in range(1, len(units)):
            sim = float(np.dot(vecs[i-1], vecs[i]))
            if (len(curr) >= min_chars and sim < sim_threshold) or (len(curr) + len(units[i]) > max_chars):
                out.append({"chunk_id": f"{doc_id}#sem{idx}", "doc_id": doc_id, "text": curr})
                curr, idx = units[i], idx + 1
            else:
                curr = f"{curr}\n{units[i]}"
        out.append({"chunk_id": f"{doc_id}#sem{idx}", "doc_id": doc_id, "text": curr})
    return out

# 3. F2 TÉCNICA B - PARENT-CHILD: Busca em filhos pequenos (350 chars), entrega a seção pai (1500 chars) (Slide 15)
def build_parent_child_chunks(docs):
    parents, children = [], []
    p_chunks = build_fixed_chunks(docs, size=1500, overlap=300)
    for p in p_chunks:
        p_id = p["chunk_id"]
        parents.append(p)
        text = p["text"]
        for c_idx, start in enumerate(range(0, len(text), 280)):
            piece = text[start:start+380].strip()
            if len(piece) > 40:
                children.append({
                    "chunk_id": f"{p_id}_c{c_idx}", "parent_id": p_id,
                    "doc_id": p["doc_id"], "text": piece, "parent_text": text
                })
    return parents, children

print("Indexando Baseline (Fixed)...")
fixed_chunks = build_fixed_chunks(docs)
fixed_emb = embed_texts([c["text"] for c in fixed_chunks])

print("Indexando F2: Semantic Chunking...")
sem_chunks = build_semantic_chunks(docs)
sem_emb = embed_texts([c["text"] for c in sem_chunks])

print("Indexando F2: Parent-Child...")
parents, child_chunks = build_parent_child_chunks(docs)
child_emb = embed_texts([c["text"] for c in child_chunks])

print("Indexando F2: BM25 (sem stopwords) para Hybrid RRF...")
bm25_tokens = [tokenize_pt(c["text"]) for c in fixed_chunks]
bm25_index = BM25Okapi(bm25_tokens)

print(f"Concluído! Fixed={len(fixed_chunks)} | Semantic={len(sem_chunks)} | Children={len(child_chunks)}")

Indexando Baseline (Fixed)...
Indexando F2: Semantic Chunking...
Indexando F2: Parent-Child...
Indexando F2: BM25 (sem stopwords) para Hybrid RRF...
Concluído! Fixed=285 | Semantic=209 | Children=1129


In [ ]:
# [Célula 5] Funções de busca (Retrievers)
def retrieve_dense(q_id, chunks_list, emb_matrix, k=TOP_K):
    t0 = time.perf_counter()
    qv = QUESTION_VECS[q_id]
    sims = emb_matrix @ qv
    top = np.argsort(-sims)[:k]
    hits = [{**chunks_list[i], "score": float(sims[i])} for i in top]
    return hits, time.perf_counter() - t0

def retrieve_parent_child(q_id, k=TOP_K):
    t0 = time.perf_counter()
    qv = QUESTION_VECS[q_id]
    sims = child_emb @ qv
    order = np.argsort(-sims)
    hits, seen_parents = [], set()
    for idx in order:
        c = child_chunks[idx]
        if c["parent_id"] not in seen_parents:
            seen_parents.add(c["parent_id"])
            hits.append({"chunk_id": c["parent_id"], "doc_id": c["doc_id"], "text": c["parent_text"], "score": float(sims[idx])})
            if len(hits) == k: break
    return hits, time.perf_counter() - t0

def retrieve_hybrid_rrf(q, k=TOP_K, rrf_k=2):
    """Combina lista Densa + lista BM25 limpa via RRF (Slides 12, 13 e 26)."""
    t0 = time.perf_counter()
    qv = QUESTION_VECS[q["id"]]
    dense_ranks = np.argsort(-(fixed_emb @ qv))

    q_toks = tokenize_pt(q["pergunta"])
    bm25_scores = bm25_index.get_scores(q_toks)
    bm25_ranks = np.argsort(-bm25_scores)

    rrf_scores = {}
    for rank, idx in enumerate(dense_ranks, start=1):
        rrf_scores[idx] = rrf_scores.get(idx, 0.0) + 1.0 / (rrf_k + rank)
    for rank, idx in enumerate(bm25_ranks, start=1):
        rrf_scores[idx] = rrf_scores.get(idx, 0.0) + 1.0 / (rrf_k + rank)

    top = sorted(rrf_scores.keys(), key=lambda i: -rrf_scores[i])[:k]
    hits = [{**fixed_chunks[i], "score": float(rrf_scores[i])} for i in top]
    return hits, time.perf_counter() - t0

In [ ]:
# ==================== CÉLULA 6 (código) ====================
# [Célula 6] Loop de avaliação e geração dos CSVs em results/
def run_experiment(name, retriever_fn, seed_base, out_filename):
    rows = []
    print(f"\n>>> Rodando {name}...")
    for q in questions:
        hits, lat_ret = retriever_fn(q)
        ans, lat_gen = generate_answer(q["pergunta"], hits, seed=seed_base + q["id"])
        score = grade_answer(q["pergunta"], q["resposta"], ans)

        expected_docs = {f["documento"] for f in q["fontes"]}
        retrieved_docs = {h["doc_id"] for h in hits}
        doc_hit = int(bool(expected_docs & retrieved_docs))

        rows.append({
            "experimento": name, "id": q["id"], "tipo": q["tipo"],
            "pergunta": q["pergunta"], "gabarito": q["resposta"],
            "resposta_modelo": ans,
            "chunks_recuperados": " | ".join(f"{h['chunk_id']} ({h['score']:.4f})" for h in hits),
            "docs_recuperados": ", ".join(dict.fromkeys(h["doc_id"] for h in hits)),
            "doc_recall": doc_hit,
            "lat_retrieval_s": round(lat_ret, 4), "lat_geracao_s": round(lat_gen, 3),
            "lat_total_s": round(lat_ret + lat_gen, 3),
            "chamadas_llm": 1, "acerto_auto": score   # avaliação automática; a manual entra na Célula 9
        })
    df = pd.DataFrame(rows)
    out_path = f"{RESULTS_DIR}/{out_filename}"
    df.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"Salvo: {out_path} | Acurácia (auto): {df['acerto_auto'].mean()*100:.1f}% | Latência média: {df['lat_total_s'].mean():.2f}s")
    return df

# Aquecimento NÃO cronometrado: na rodada anterior o baseline_r1 foi ~2,6x mais lento que o r2 em todas as
# perguntas, com a mesma recuperação (efeito de ordem/GPU fria). Descartamos esta passada.
print("Aquecimento (descartado)...")
for q in questions:
    _h, _ = retrieve_dense(q["id"], fixed_chunks, fixed_emb)
    generate_answer(q["pergunta"], _h, seed=0)

# 3 rodadas do Baseline (r1, r2, r3)
b1 = run_experiment("baseline_r1", lambda q: retrieve_dense(q["id"], fixed_chunks, fixed_emb), 1000, "baseline_r1.csv")
b2 = run_experiment("baseline_r2", lambda q: retrieve_dense(q["id"], fixed_chunks, fixed_emb), 2000, "baseline_r2.csv")
b3 = run_experiment("baseline_r3", lambda q: retrieve_dense(q["id"], fixed_chunks, fixed_emb), 3000, "baseline_r3.csv")

# 3 técnicas da Família F2
f2_sem = run_experiment("semantic_chunking", lambda q: retrieve_dense(q["id"], sem_chunks, sem_emb), 4000, "results_semantic_chunking.csv")
f2_pc  = run_experiment("parent_child", lambda q: retrieve_parent_child(q["id"]), 5000, "results_parent_child.csv")
f2_hyb = run_experiment("hybrid_rrf", lambda q: retrieve_hybrid_rrf(q), 6000, "results_hybrid_rrf.csv")

Aquecimento (descartado)...

>>> Rodando baseline_r1...
Salvo: advanced_rag/results/baseline_r1.csv | Acurácia (auto): 75.0% | Latência média: 2.43s

>>> Rodando baseline_r2...
Salvo: advanced_rag/results/baseline_r2.csv | Acurácia (auto): 80.0% | Latência média: 2.16s

>>> Rodando baseline_r3...
Salvo: advanced_rag/results/baseline_r3.csv | Acurácia (auto): 60.0% | Latência média: 2.29s

>>> Rodando semantic_chunking...
Salvo: advanced_rag/results/results_semantic_chunking.csv | Acurácia (auto): 55.0% | Latência média: 3.38s

>>> Rodando parent_child...
Salvo: advanced_rag/results/results_parent_child.csv | Acurácia (auto): 75.0% | Latência média: 3.73s

>>> Rodando hybrid_rrf...
Salvo: advanced_rag/results/results_hybrid_rrf.csv | Acurácia (auto): 85.0% | Latência média: 2.88s


In [ ]:
# ==================== CÉLULA 7 (código) ====================
# [Célula 7] Consolidação (avaliação AUTOMÁTICA), planilha CEGA para avaliação manual e download do ZIP
all_dfs = [b1, b2, b3, f2_sem, f2_pc, f2_hyb]
df_all = pd.concat(all_dfs, ignore_index=True)

# 1. Perguntas que mudam entre as 3 rodadas do Baseline (ruído do gerador)
piv_base = pd.concat([b1, b2, b3]).pivot(index=["id", "tipo"], columns="experimento", values="acerto_auto")
piv_base["variou"] = piv_base.nunique(axis=1) > 1
print("=== DIAGNÓSTICO DAS 3 RODADAS DO BASELINE (acerto automático) ===")
print(piv_base.to_string())

# 2. Tabela comparativa AUTOMÁTICA (a tabela final, com o acerto manual, é gerada na Célula 9)
summary = df_all.groupby("experimento").agg(
    acuracia_auto=("acerto_auto", lambda s: round(s.mean() * 100, 1)),
    recall_doc=("doc_recall", lambda s: round(s.mean() * 100, 1)),
    chamadas_llm_por_q=("chamadas_llm", "mean"),
    latencia_media_s=("lat_total_s", "mean")
).round(2)
tiers = df_all.pivot_table(index="experimento", columns="tipo", values="acerto_auto", aggfunc=lambda s: round(s.mean()*100, 1))
auto_table = pd.concat([summary, tiers], axis=1)
auto_table.to_csv(f"{RESULTS_DIR}/tabela_comparativa_auto.csv", encoding="utf-8-sig")
print("\n=== TABELA COMPARATIVA (acerto AUTOMÁTICO) ===")
print(auto_table.to_string())

# 3. Planilha CEGA para avaliação manual: sem nome do experimento e sem a nota automática.
#    Linhas agrupadas por pergunta (mesmo gabarito), com a ordem dos sistemas embaralhada.
blind = df_all[["experimento", "id", "tipo", "pergunta", "gabarito", "resposta_modelo"]].copy()
blind = blind.sample(frac=1, random_state=42).sort_values("id", kind="stable").reset_index(drop=True)
blind.insert(0, "id_cego", [f"A{i+1:03d}" for i in range(len(blind))])
blind[["id_cego", "experimento", "id"]].to_csv(f"{RESULTS_DIR}/chave_cega.csv", index=False, encoding="utf-8-sig")
sheet = blind[["id_cego", "tipo", "pergunta", "gabarito", "resposta_modelo"]].copy()
sheet["acerto_manual"] = ""
sheet.to_csv(f"{RESULTS_DIR}/avaliacao_manual_cega.csv", index=False, encoding="utf-8-sig")
print(f"\nPlanilha cega gerada: {len(sheet)} linhas em {RESULTS_DIR}/avaliacao_manual_cega.csv")

# 4. Baixar os CSVs gerados em um único ZIP
from google.colab import files
shutil.make_archive("resultados_f2_parcial", "zip", RESULTS_DIR)
files.download("resultados_f2_parcial.zip")

=== DIAGNÓSTICO DAS 3 RODADAS DO BASELINE (acerto automático) ===
experimento          baseline_r1  baseline_r2  baseline_r3  variou
id tipo                                                           
1  termo-exato                 1            1            1   False
2  termo-exato                 0            1            0    True
3  multi-chunk                 1            1            0    True
4  multi-chunk                 1            1            1   False
5  simples                     1            1            1   False
6  simples                     1            1            1   False
7  termo-exato                 0            0            0   False
8  multi-chunk                 1            0            0    True
9  simples                     0            1            1    True
10 simples                     1            1            1   False
11 simples                     1            1            0    True
12 multi-chunk                 1            1            1   Fa

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# ==================== CÉLULA 8 ====================
# ### Critério de avaliação manual (definido ANTES de avaliar)
# `acerto_manual = 1` se a resposta traz **todos os elementos que a pergunta pede** (números, prazos,
# regras e exceções do gabarito pertinentes à pergunta) e **não contradiz** o gabarito; `0` caso contrário.
# Resposta vazia ou `INFORMAÇÃO AUSENTE` = 0. Citação de artigo incorreta não invalida a resposta
# (é registrada à parte). A planilha é cega: não mostra o experimento nem a nota automática.
# Preencha `acerto_manual` (0 ou 1) em `results/avaliacao_manual_cega.csv` e suba o arquivo para `results/`.

In [ ]:
# ==================== CÉLULA 9 (código) ====================
# [Célula 9] Mescla a avaliação manual (planilha cega preenchida) e gera a tabela comparativa FINAL
# Pré-requisito: o arquivo avaliacao_manual_cega.csv preenchido (acerto_manual = 0 ou 1) em results/.
import os, glob, shutil
import numpy as np
import pandas as pd
RESULTS_DIR = globals().get("RESULTS_DIR", "advanced_rag/results")

def read_csv_flex(path):
    """Lê CSV salvo pelo Excel/Sheets: detecta separador (',' ou ';') e tenta UTF-8 antes de Latin-1."""
    for enc in ("utf-8-sig", "latin-1"):
        try:
            return pd.read_csv(path, sep=None, engine="python", encoding=enc)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Não foi possível ler {path}")

sheet = read_csv_flex(f"{RESULTS_DIR}/avaliacao_manual_cega.csv")
key = read_csv_flex(f"{RESULTS_DIR}/chave_cega.csv")
sheet["acerto_manual"] = pd.to_numeric(sheet["acerto_manual"], errors="coerce")
assert sheet["acerto_manual"].isin([0, 1]).all(), "Há linhas sem nota (0 ou 1) em acerto_manual"
manual = sheet[["id_cego", "acerto_manual"]].merge(key, on="id_cego", validate="1:1")
assert len(manual) == len(sheet) == len(key), "A planilha e a chave não têm o mesmo número de linhas"

dfs = []
paths = sorted(glob.glob(f"{RESULTS_DIR}/baseline_r*.csv") + glob.glob(f"{RESULTS_DIR}/results_*.csv"))
for path in paths:
    df = pd.read_csv(path, encoding="utf-8-sig")
    df = df.drop(columns=["acerto_manual"], errors="ignore").merge(
        manual[["experimento", "id", "acerto_manual"]], on=["experimento", "id"], how="left", validate="1:1")
    assert df["acerto_manual"].notna().all(), f"Faltam notas manuais em {path}"
    df["acerto_manual"] = df["acerto_manual"].astype(int)
    df.to_csv(path, index=False, encoding="utf-8-sig")
    dfs.append(df)
df_all = pd.concat(dfs, ignore_index=True)

pct = lambda s: round(s.mean() * 100, 1)
summary = df_all.groupby("experimento").agg(
    acuracia_manual=("acerto_manual", pct),
    acuracia_auto=("acerto_auto", pct),
    recall_doc=("doc_recall", pct),
    chamadas_llm_por_q=("chamadas_llm", "mean"),
    latencia_media_s=("lat_total_s", "mean"),
).round(2)
tiers = df_all.pivot_table(index="experimento", columns="tipo", values="acerto_manual", aggfunc=pct)
final_table = pd.concat([summary, tiers], axis=1)
final_table.to_csv(f"{RESULTS_DIR}/tabela_comparativa_final.csv", encoding="utf-8-sig")

print("=== TABELA COMPARATIVA FINAL (acerto MANUAL como métrica principal) ===")
print(final_table.to_string())
base = final_table.loc[final_table.index.str.startswith("baseline_r")]
print("\nBaseline (média das 3 rodadas):", base.mean(numeric_only=True).round(2).to_dict())
print("Concordância auto × manual:", int((df_all["acerto_auto"] == df_all["acerto_manual"]).sum()), "de", len(df_all))

try:
    from google.colab import files
    shutil.make_archive("resultados_f2_final", "zip", RESULTS_DIR)
    files.download("resultados_f2_final.zip")
except ImportError:
    pass

=== TABELA COMPARATIVA FINAL (acerto MANUAL como métrica principal) ===
                   acuracia_manual  acuracia_auto  recall_doc  chamadas_llm_por_q  latencia_media_s  entre-documentos  multi-chunk  simples  termo-exato
experimento                                                                                                                                             
baseline_r1                   50.0           75.0       100.0                 1.0              2.43              60.0         33.3     60.0         50.0
baseline_r2                   50.0           80.0       100.0                 1.0              2.16              60.0         50.0     40.0         50.0
baseline_r3                   45.0           60.0       100.0                 1.0              2.29              60.0         50.0     40.0         25.0
hybrid_rrf                    55.0           85.0       100.0                 1.0              2.88              60.0         50.0     40.0         75.0
parent_chi

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>